In [ ]:
import numpy as np
import glob
import re
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size" : 12,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
})



In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec

def read_col_csv(filePath, colIdx, skipHeader):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx, delimiter=',')
    f.close()
    return vec

In [ ]:
folder = "/Users/maggie/repo/LBM-Program/amrlbm/bin/cavity2dRe400_cas_100k_rf0/probes/probe_X_V"
# folder = "/Users/maggie/repo/LBM-Program/amrlbm/bin/cavity2dRe400_cas_100k_rf0/probes/probe_Y_U"
fileList = sorted(glob.glob(f"{folder}/probe*.txt"))

ghiaYUPath = "../data/2d/ghia-400-y-u.dat"
ghiaVXPath = "../data/2d/ghia-400-v-x.dat"

U0 = 0.05

colIdx = 3
coordStr = 'x'

In [ ]:

coordVec = []
dataVec = []

coordMap = {
    'x': 0,
    'y': 1,
    'z': 2
}
coordIdx = coordMap[coordStr]

coord_pattern = re.compile(r"\((.*?)\)")
for file_path in fileList:
    with open(file_path, 'r') as f:
        lines = f.readlines()
        if not lines:
            continue

        first_line = lines[0].strip()
        match = coord_pattern.search(first_line)
        if match:
            coords = match.group(1).split(',')
            coordVec.append(float(coords[coordIdx]))
        
        last_line = lines[-1].strip()
        columns = last_line.split()
        if len(columns) >= (colIdx + 1):
            dataVec.append(float(columns[colIdx])/U0)

ghiaCoordVec = []
ghiaDataVec = []

if coordStr == 'x':
    ghiaCoordVec = read_col_csv(ghiaVXPath, 0, 1)
    ghiaDataVec = read_col_csv(ghiaVXPath, 1, 1)
elif coordStr == 'y':
    ghiaCoordVec = read_col_csv(ghiaYUPath, 1, 1)
    ghiaDataVec = read_col_csv(ghiaYUPath, 0, 1)

fig, ax = plt.subplots(1, 1, figsize=(6, 5), facecolor='w', edgecolor='w')
if coordStr == 'x':
    ax.plot(coordVec, dataVec, marker='o', fillstyle='none', linestyle='-', color='#910000', label='LBM Cascaded (Probe) (Re=400)')
    ax.plot(ghiaCoordVec, ghiaDataVec, marker='s', c='k', lw=0, label='Ghia et al. (Re=400)')
    ax.set_xlabel('X')
    ax.set_ylabel(r'$V/U_0$')
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.6, 0.6)
    ax.legend(loc='upper right', frameon=False)
elif coordStr == 'y':
    ax.plot(dataVec, coordVec, marker='o', fillstyle='none', linestyle='-', color='#910000', label='LBM Cascaded (Probe) (Re=400)')
    ax.plot(ghiaDataVec, ghiaCoordVec, marker='s', c='k', lw=0, label='Ghia et al. (Re=400)')
    ax.set_xlabel(r'$U/U_0$')
    ax.set_ylabel('Y')
    ax.set_xlim(-0.6, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc='lower right', frameon=False)

